In [1]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
# sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp", spark_submit="spark3-submit")
 
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-02-06 09:56:51 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/ADQUIRENCIA_FERIA_EVA/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



In [ ]:
# Query inicial

sql = """
-- EXPLAIN
WITH ult_ing AS
  (SELECT ingestion_year,
          ingestion_month,
          MAX(ingestion_day) AS ingestion_day
   FROM resultados_vspc_medios_de_pago.gsap_m_comercios
   GROUP BY 1,
            2
   ORDER BY ingestion_year DESC, ingestion_month DESC),
     info AS
  (SELECT id_comercio_padre,
          id_comercio AS cod_unico,
          com.f_vinculacion_opy -- com.ingestion_month, com.ingestion_year

   FROM resultados_vspc_medios_de_pago.gsap_m_comercios AS com
   LEFT JOIN ult_ing AS ult ON com.ingestion_year = ult.ingestion_year
   AND com.ingestion_month = ult.ingestion_month
   AND com.ingestion_day = ult.ingestion_day
   WHERE (id_comercio_padre = '16420697'
          OR id_comercio_padre = '16249062'
          OR id_comercio_padre = '16243073'
          OR id_comercio_padre = '15021702'
          OR id_comercio_padre = '15026362'
          OR id_comercio_padre = '19245380'
          OR id_comercio_padre = '19747609'
          OR id_comercio_padre = '19149806'
          OR id_comercio_padre = '20657045'
          OR id_comercio_padre = '20750808'
          OR id_comercio_padre = '18965848'
          OR id_comercio_padre = '22498349'
          OR id_comercio_padre = '19969146'
          OR id_comercio_padre = '22701601'
          OR id_comercio_padre = '22845325'
          OR id_comercio_padre = '22784474'
          OR id_comercio_padre = '22756944'
          OR id_comercio_padre = '22756902'
          OR id_comercio_padre = '22837413'
          OR id_comercio_padre = '22831580')
     AND UPPER(estado_comercio) LIKE 'ACTIV%%' ),
     transantes AS
  (SELECT inf.id_comercio_padre, -- inf.ingestion_year as año,
 -- inf.ingestion_month as mes,
 inf.f_vinculacion_opy,
 year(trx.f_trx) AS trx_year,
 month(trx.f_trx) AS trx_month,
 count(distinct(inf.cod_unico)) AS cantidad_transantes,
 count(distinct(trx.cod_unico)) AS cant_trx
   FROM resultados_vspc_medios_de_pago.gsap_m_transaccional AS trx
   RIGHT JOIN info AS inf ON trx.cod_unico = inf.cod_unico -- AND trx.ingestion_month = inf.ingestion_month

   WHERE trx.ingestion_year = 2025
   GROUP BY 1,
            2,
            3,
            4),
     activos AS
  (SELECT id_comercio_padre, -- ingestion_month,
 -- ingestion_year,
 count(distinct(cod_unico)) AS cantidad_activos
   FROM info
   GROUP BY 1--,2,3
)
SELECT distinct(trx.id_comercio_padre) AS cod_unico,
       trx.trx_year,
       trx.trx_month, -- trx.f_vinculacion_opy,
--trx.id_comercio_padre,
ac.cantidad_activos,
                         trx.cantidad_transantes
FROM transantes trx
LEFT JOIN activos ac ON trx.id_comercio_padre = ac.id_comercio_padre -- AND trx.mes = ac.ingestion_month
ORDER BY trx.trx_year DESC,
         trx.trx_month DESC ;
"""